In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.config import RAW_DIR

comp = pd.read_csv(RAW_DIR / "competitions.csv")
print(comp[["competition_id", "name", "country_name", "type"]].to_string())

trf = pd.read_csv(RAW_DIR / "transfers.csv")
print(trf.shape, trf["transfer_date"].min(), trf["transfer_date"].max())
print(trf.head())

   competition_id                                     name    country_name                       type
0              A1                               bundesliga         Austria            domestic_league
1            AFAC                            afc-asian-cup             NaN  national_team_competition
2            AFCN                    africa-cup-of-nations             NaN  national_team_competition
3            ARG1                          torneo-apertura       Argentina            domestic_league
4            AUS1                             a-league-men       Australia            domestic_league
5             BE1                       jupiler-pro-league         Belgium            domestic_league
6            BESC                      volkswagen-supercup         Belgium                      other
7            BPO4  jupiler-pro-league-relegation-play-offs         Belgium                      other
8            BRA1            campeonato-brasileiro-serie-a          Brazil        

In [2]:
# ---- 1. League coverage: which seasons each competition has ----

# Load only the two columns we need from games.csv (faster, less memory)
games = pd.read_csv(RAW_DIR / "games.csv", usecols=["competition_id", "season"])

# For each competition, find its first season, last season, and how many seasons it has
cov = games.groupby("competition_id")["season"].agg(["min", "max", "nunique"])

# Print the full table (to_string stops pandas hiding rows with "...")
print(cov.to_string())


# ---- 2. Data quality checks on transfers ----

# Convert transfer_date from text to real dates so we can compare them
trf["transfer_date"] = pd.to_datetime(trf["transfer_date"])

# The dataset's snapshot date: anything after this hasn't actually happened
snap = pd.Timestamp("2026-07-06")

# Count transfers dated in the future (these must be dropped later)
print("future dated:", (trf["transfer_date"] > snap).sum())

# Count missing fees vs fees recorded as exactly 0
# (tells us if 0 means "free/loan" or if unknowns are stored as 0)
print("fee null:", trf["transfer_fee"].isna().sum(), "fee zero:", (trf["transfer_fee"] == 0).sum())


# ---- 3. Rough count of transfers INTO the top five leagues ----

# Load each club's league (NOTE: this is the club's CURRENT league, not historical)
clubs = pd.read_csv(RAW_DIR / "clubs.csv", usecols=["club_id", "domestic_competition_id"])

# Destination leagues: England, Spain, Italy, Germany, France
top5 = {"GB1", "ES1", "IT1", "L1", "FR1"}

# Keep only transfers that have actually happened
t = trf[trf["transfer_date"] <= snap]

# Attach the SELLING club's league: rename clubs columns so they match
# from_club_id, then join (how="left" keeps transfers even if no club match)
t = t.merge(clubs.rename(columns={"club_id": "from_club_id", "domestic_competition_id": "from_comp"}),
            on="from_club_id", how="left")

# Same again for the BUYING club's league
t = t.merge(clubs.rename(columns={"club_id": "to_club_id", "domestic_competition_id": "to_comp"}),
            on="to_club_id", how="left")

# "Movers" = joined a top five club, came from a known league that is NOT top five
movers = t[t["to_comp"].isin(top5)          # destination is top five
           & t["from_comp"].notna()          # source league is known
           & ~t["from_comp"].isin(top5)]     # source is not top five (~ means NOT)

# How many movers came from each source league
print(movers["from_comp"].value_counts().to_string())

# How many movers per year (checks we have enough history to backtest)
print(movers.groupby(movers["transfer_date"].dt.year).size().to_string())

                 min   max  nunique
competition_id                     
A1              2024  2025        2
AFAC            2024  2024        1
AFCN            2025  2025        1
ARG1            2024  2025        2
AUS1            2024  2025        2
BE1             2012  2025       14
BESC            2012  2025       13
BPO4            2025  2025        1
BRA1            2024  2025        2
C1              2024  2025        2
CDR             2012  2025       14
CGB             2022  2025        3
CIT             2012  2025       14
CL              2012  2025       14
CLQ             2012  2025       14
COL1            2025  2025        1
COPA            2025  2025        1
DFB             2012  2025       14
DFL             2012  2025       14
DK1             2012  2025       14
DKP             2012  2025       14
ECLQ            2021  2025        5
EJPL            2025  2025        1
EL              2012  2025       14
ELQ             2012  2025       14
ES1             2012  2025  